In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os
import shutil
from concurrent.futures import ThreadPoolExecutor
from tqdm.notebook import tqdm

src_base = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/raw"
dst_base = "/content/KAGGLE_TMP"

# Lấy danh sách tất cả các thư mục LIDC-IDRI và sắp xếp theo tên
all_patients = sorted([d for d in os.listdir(src_base) if os.path.isdir(os.path.join(src_base, d))])
part1_patients = all_patients[:505] # Lấy 505 ca đầu tiên

print(f"🔍 Đang quét file của {len(part1_patients)} ca bệnh phần 1...")
files_to_copy = []
for patient in tqdm(part1_patients):
    p_path = os.path.join(src_base, patient)
    for root, _, files in os.walk(p_path):
        for f in files:
            files_to_copy.append(os.path.join(root, f))

print(f"🚀 Bắt đầu copy {len(files_to_copy)} file bằng 64 luồng mạng (Multi-threading)...")

def copy_file(src):
    dst = src.replace(src_base, dst_base)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

# Dùng 64 luồng ép Google Drive nhả băng thông
with ThreadPoolExecutor(max_workers=64) as executor:
    list(tqdm(executor.map(copy_file, files_to_copy), total=len(files_to_copy)))

print("✅ Đã copy xong Part 1 xuống SSD!")

✅ Đã cấu hình xong Kaggle API từ thư mục content!


In [6]:
# 1. Tạo thư mục tạm trên ổ SSD của Colab
!mkdir -p /content/KAGGLE_TMP

# 2. Dùng rsync để copy (Có thanh hiển thị % và tốc độ)
!echo "📥 Đang copy các thư mục LIDC-IDRI từ Drive xuống SSD cục bộ..."
!echo "⏳ Google Drive đọc file nhỏ khá chậm, vui lòng đợi..."

# 🛑 SỬA ĐƯỜNG DẪN TẠI ĐÂY (Giữ nguyên dấu / ở cuối):
!rsync -ah --info=progress2 "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/SOURCES/data/raw/" /content/KAGGLE_TMP/

!echo "✅ Dữ liệu đã copy xong và sẵn sàng trên ổ SSD!"

📥 Đang copy các thư mục LIDC-IDRI từ Drive xuống SSD cục bộ...
⏳ Google Drive đọc file nhỏ khá chậm, vui lòng đợi...
          2.10G  79%    1.56MB/s    0:05:35  
rsync error: received SIGINT, SIGTERM, or SIGHUP (code 20) at rsync.c(716) [sender=3.2.7]
rsync error: received SIGUSR1 (code 19) at main.c(1635) [generator=3.2.7]
✅ Dữ liệu đã copy xong và sẵn sàng trên ổ SSD!


In [ ]:
import os
import json

# Chuyển hướng làm việc vào thư mục SSD
os.chdir('/content/KAGGLE_TMP')

# Lệnh của Kaggle để tạo file dataset-metadata.json mặc định
!kaggle datasets init

# Đọc và sửa file bằng Python
with open('dataset-metadata.json', 'r') as f:
    meta = json.load(f)

# 🛑 SỬA TẠI ĐÂY (Chỉ dùng chữ cái thường, số và dấu gạch ngang):
meta['id'] = 'minhtri2475/lidc-idri-raw-data-ct208e'
meta['title'] = 'LIDC-IDRI Raw Data CT208E'

# Ghi đè file
with open('dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=4)

print("✅ Đã cập nhật metadata thành công!")

In [ ]:
%cd /content/KAGGLE_TMP
!echo "🚀 Bắt đầu nén và đẩy dữ liệu từ SSD Colab lên máy chủ Kaggle..."

# Lệnh tạo Dataset mới:
!kaggle datasets create -p .